In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path(__file__).resolve().parents[2]
RAW_DIR = BASE_DIR / "ml" / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "ml" / "data" / "processed"
PLOT_DIR = OUTPUT_DIR / "eda_plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FILE_PATH = RAW_DIR / "perishable_inventory.csv"

if not FILE_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {FILE_PATH}")

df = pd.read_csv(FILE_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date")

sns.set_theme(style="whitegrid")

print("=" * 60)
print("WASTEWISE AI - EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("\nDataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumerical summary:")
print(df.describe().T)

print("\nCategory counts:")
print(df["category"].value_counts())

print("\nTotal units sold:", df["units_sold"].sum())
print("Total waste units:", df["waste_units"].sum())

total_sold = df["units_sold"].sum()
total_waste = df["waste_units"].sum()

if total_sold + total_waste > 0:
    waste_percentage = total_waste / (total_sold + total_waste) * 100
else:
    waste_percentage = 0

print("Waste percentage of sold plus wasted units:",
      round(waste_percentage, 2), "%")


def save_plot(filename):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# 1. Units sold distribution
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="units_sold", bins=25, kde=True)
plt.title("Distribution of Units Sold")
plt.xlabel("Units Sold")
plt.ylabel("Number of Records")
save_plot("01_units_sold_distribution.png")


# 2. Total waste by product
waste_product = (
    df.groupby("product_id", as_index=False)["waste_units"]
    .sum()
    .sort_values("waste_units", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=waste_product,
    x="product_id",
    y="waste_units"
)
plt.title("Total Food Waste by Product")
plt.xlabel("Product ID")
plt.ylabel("Total Waste Units")
save_plot("02_waste_by_product.png")


# 3. Total waste by category
waste_category = (
    df.groupby("category", as_index=False)["waste_units"]
    .sum()
    .sort_values("waste_units", ascending=False)
)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=waste_category,
    x="category",
    y="waste_units"
)
plt.title("Total Food Waste by Category")
plt.xlabel("Product Category")
plt.ylabel("Total Waste Units")
plt.xticks(rotation=30)
save_plot("03_waste_by_category.png")


# 4. Daily sales and waste trends
daily = (
    df.groupby("date", as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        waste_units=("waste_units", "sum")
    )
    .sort_values("date")
)

plt.figure(figsize=(12, 5))
plt.plot(
    daily["date"], daily["units_sold"],
    label="Units Sold"
)
plt.plot(
    daily["date"], daily["waste_units"],
    label="Waste Units"
)
plt.title("Daily Sales and Food Waste Trends")
plt.xlabel("Date")
plt.ylabel("Units")
plt.legend()
plt.xticks(rotation=45)
save_plot("04_daily_sales_and_waste.png")


# 5. Waste by days to expiry
expiry_waste = (
    df.groupby("days_to_expiry", as_index=False)
    .agg(
        average_waste=("waste_units", "mean"),
        total_waste=("waste_units", "sum")
    )
    .sort_values("days_to_expiry")
)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=expiry_waste,
    x="days_to_expiry",
    y="average_waste",
    marker="o"
)
plt.title("Average Waste by Days to Expiry")
plt.xlabel("Days to Expiry")
plt.ylabel("Average Waste Units")
save_plot("05_waste_by_expiry.png")


# 6. Discount percentage versus units sold
plt.figure(figsize=(9, 5))
sns.barplot(
    data=df,
    x="discount_pct",
    y="units_sold",
    errorbar=None
)
plt.title("Average Units Sold by Discount Percentage")
plt.xlabel("Discount Percentage")
plt.ylabel("Average Units Sold")
save_plot("06_discount_vs_sales.png")


# 7. Correlation heatmap
numeric_df = df.select_dtypes(include=np.number)
correlation = numeric_df.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True
)
plt.title("Correlation Between Numerical Features")
save_plot("07_correlation_heatmap.png")


# 8. Waste occurrence by category
df["waste_occurred"] = (df["waste_units"] > 0).astype(int)

waste_occurrence = (
    df.groupby("category", as_index=False)
    .agg(
        records=("waste_occurred", "size"),
        waste_records=("waste_occurred", "sum")
    )
)

waste_occurrence["waste_rate_pct"] = (
    waste_occurrence["waste_records"]
    / waste_occurrence["records"] * 100
)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=waste_occurrence,
    x="category",
    y="waste_rate_pct"
)
plt.title("Percentage of Records with Waste by Category")
plt.xlabel("Category")
plt.ylabel("Records with Waste (%)")
plt.xticks(rotation=30)
save_plot("08_waste_occurrence_by_category.png")


# 9. Stockout analysis
stockout_summary = (
    df.groupby("stockout_flag", as_index=False)
    .agg(
        average_sales=("units_sold", "mean"),
        average_waste=("waste_units", "mean"),
        records=("product_id", "size")
    )
)

stockout_summary["stockout_status"] = stockout_summary[
    "stockout_flag"
].map({0: "No Stockout", 1: "Stockout"})

plt.figure(figsize=(8, 5))
sns.barplot(
    data=stockout_summary,
    x="stockout_status",
    y="average_sales"
)
plt.title("Average Sales by Stockout Status")
plt.xlabel("Stockout Status")
plt.ylabel("Average Units Sold")
save_plot("09_sales_by_stockout.png")


# 10. Save EDA summary reports
waste_product.to_csv(
    OUTPUT_DIR / "eda_waste_by_product.csv",
    index=False
)

waste_category.to_csv(
    OUTPUT_DIR / "eda_waste_by_category.csv",
    index=False
)

daily.to_csv(
    OUTPUT_DIR / "eda_daily_sales_and_waste.csv",
    index=False
)

expiry_waste.to_csv(
    OUTPUT_DIR / "eda_waste_by_expiry.csv",
    index=False
)

waste_occurrence.to_csv(
    OUTPUT_DIR / "eda_waste_occurrence_by_category.csv",
    index=False
)

stockout_summary.to_csv(
    OUTPUT_DIR / "eda_stockout_summary.csv",
    index=False
)

print("\n" + "=" * 60)
print("EDA SUMMARY")
print("=" * 60)

print("\nWaste by product:")
print(waste_product.to_string(index=False))

print("\nWaste by category:")
print(waste_category.to_string(index=False))

print("\nWaste occurrence by category:")
print(waste_occurrence.to_string(index=False))

print("\nAll 9 visualizations saved in:")
print(PLOT_DIR)

print("\nEDA report files saved in:")
print(OUTPUT_DIR)

print("\nEXPLORATORY DATA ANALYSIS COMPLETED SUCCESSFULLY")
